# FXS / 01: stacked 1-minute days — discovery scan 2016–2020

Protocol: [`001_claude_fxs01_preregistration.md`](../../discussions/FX_Intraday_Seasonality/001_claude_fxs01_preregistration.md).

Every trading day is laid on top of the others by minute of day, in two clocks (`NY`: 17:00→17:00 New York; `UTC`: 00:00→24:00). Every interval (start every 5 minutes, 15–240 minutes long; 40,560 tests over 15 pairs) is tested for a non-zero mean.

| Filter | Condition |
|---|---|
| D1 | Benjamini–Hochberg over all tests, q ≤ 0.05 |
| D2 | same sign in ≥ 4 of 5 years |
| D3 | \|mean\| > median spread (start, end) + 0.70 bp commission |
| D0 (global) | real D1–D3 count above the 95th percentile of 20 rotated-day placebos |

**This notebook reads 2016–2020 only** (enforced in code). It writes `01/candidates_frozen.csv`; commit it before running notebook 02.

`FXS_SYNTHETIC=1`: three synthetic pairs, EURUSD with +4 bp/day injected between 08:00 and 09:00 New York.

In [ ]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace, asdict
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs_")
    os.environ["FXS_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F

cfg = F.FXSConfig()
PAIRS = list(F.PAIRS)
if SYNTHETIC:
    PAIRS = ["EURUSD", "USDJPY", "CADCHF"]
    cfg = replace(cfg, n_placebo=3, n_boot=2_000)
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_rows", 120)
OUT = Path.cwd() / ("01_synthetic" if SYNTHETIC else "01"); OUT.mkdir(exist_ok=True)
START, END = F.DISCOVERY
print(f"fx_seasonality {F.FXS_VERSION} | synthetic={SYNTHETIC} | {START} -> {END} | {len(PAIRS)} pairs | "
      f"{len(F.intervals(cfg))} intervals per pair and clock")

## 0. Synthetic inputs (smoke run only)

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        for a, b in [(dt.date(2015, 12, 28), dt.date(2020, 12, 31)), (dt.date(2020, 12, 28), dt.date(2025, 12, 31))]:
            F.write_synthetic_candles(base, p, a, b, seed=10 * i + a.year, drift_bps=4.0 if p == "EURUSD" else 0.0,
                                      price=110.0 if p.endswith("JPY") else 1.10)

## 1. Candle format check against the USD/JPY ticks

The downloader checks every file's layout; this confirms the minute **close** against the last tick of each minute, on the Tokyo-morning ticks already downloaded for the gotobi study (2016 days). Expect ≈ 100% matches; a low match rate means the close field is wrong and nothing below can be trusted.

In [ ]:
tick_root = F.candles_root("USDJPY").parent / "hours_00-01-02-22-23"
if not SYNTHETIC and tick_root.exists():
    val = F.validate_against_ticks("USDJPY", F.candles_root("USDJPY"), tick_root,
                                   [dt.date(2016, 3, d) for d in range(1, 32) if dt.date(2016, 3, d).weekday() < 5])
    display(val.describe().loc[["count", "mean", "min"]])
    assert len(val) and val["bid_close_match"].mean() > 0.95, "candle close does not match ticks"
else:
    print("tick folder not available (or synthetic run): skipped")

## 2. Load and stack (2016–2020)

First run reads every day file (~1,300 per pair) and caches the minute series under `<EQUITY_TRADING_DIR>/EquityTrading/FXSeasonality/`.

In [ ]:
panels = F.load_panels(PAIRS, START, END, "discovery", cfg)
cov = pd.DataFrame([{"pair": p, "clock": c, "days": len(v.dates), "first": v.dates[0], "last": v.dates[-1],
                     "median_coverage": float(np.median(v.coverage)),
                     "median_spread_bps": float(np.median(v.spread_bps()))} for (p, c), v in panels.items()])
display(cov.pivot(index="pair", columns="clock", values=["days", "median_coverage", "median_spread_bps"]).round(3))
cov.to_csv(OUT / "01_coverage.csv", index=False)

## 3. The stacked day: average cumulative return

Mean cumulative mid return from the start of the day (bps) with a ±2 standard-error band. A flat line inside the band is no structure; a band that leaves zero and stays out marks a candidate. Shaded: hours where the median spread is above twice its daily median (rollover, thin hours).

In [ ]:
prof = {k: F.profile(v) for k, v in panels.items()}
for clock in F.CLOCKS:
    fig, axes = plt.subplots(5, 3, figsize=(16, 15), sharex=True)
    for ax, p in zip(axes.flat, PAIRS):
        pr = prof[(p, clock)]
        x = pr.index.to_numpy() / 60
        ax.plot(x, pr["cum_mean_bps"], lw=1)
        ax.fill_between(x, pr["cum_mean_bps"] - 2 * pr["cum_se_bps"], pr["cum_mean_bps"] + 2 * pr["cum_se_bps"], alpha=0.25)
        wide = pr["spread_bps"] > 2 * pr["spread_bps"].median()
        ax.fill_between(x, 0, 1, where=wide.to_numpy(), transform=ax.get_xaxis_transform(), color="grey", alpha=0.15)
        ax.axhline(0, color="black", lw=0.5); ax.set_title(p)
        ax.set_xticks(range(0, 25, 4))
        ax.set_xticklabels([F.slot_label(clock, 60 * h).split()[1] for h in range(0, 25, 4)])
    for ax in axes.flat[len(PAIRS):]:
        ax.axis("off")
    fig.suptitle(f"Stacked day, {clock} clock, {START.year}-{END.year}: cumulative mean return (bps) ± 2 SE")
    plt.tight_layout(); plt.savefig(OUT / f"01_profile_{clock}.png", dpi=110); plt.show()
pd.concat({f"{p}|{c}": pr["cum_mean_bps"] for (p, c), pr in prof.items()}, axis=1).to_csv(OUT / "01_profile_cum_mean.csv")

## 4. Activity and cost by minute of day

Mean absolute 1-minute return and median spread (bps). Structure in these two is certain (sessions, rollover) and is not a trade signal; it tells where an interval is cheap enough to trade.

In [ ]:
for clock in F.CLOCKS:
    fig, axes = plt.subplots(1, 2, figsize=(16, 4))
    for p in PAIRS:
        pr = prof[(p, clock)]
        axes[0].plot(pr.index / 60, pr["abs_ret_bps"].rolling(15, min_periods=1).mean(), lw=0.8, label=p)
        axes[1].plot(pr.index / 60, pr["spread_bps"].rolling(15, min_periods=1).median(), lw=0.8, label=p)
    axes[0].set_title(f"{clock}: mean |1-min return| (bps, 15-min smoothing)")
    axes[1].set_title(f"{clock}: median spread (bps)"); axes[1].set_yscale("log")
    for ax in axes:
        ax.set_xticks(range(0, 25, 2)); ax.set_xticklabels([F.slot_label(clock, 60 * h).split()[1] for h in range(0, 25, 2)], rotation=45)
    axes[1].legend(ncol=3, fontsize=7)
    plt.tight_layout(); plt.savefig(OUT / f"01_activity_{clock}.png", dpi=110); plt.show()

## 5. Hour by hour, year by year

t-statistic of each one-hour block over 2016–2020 (left) and the number of years whose mean has the same sign as the 5-year mean (right). Descriptive only; the formal test is the interval scan below.

In [ ]:
for clock in F.CLOCKS:
    hs = pd.concat([F.hourly_stats(panels[(p, clock)]).assign(pair=p) for p in PAIRS])
    allt = hs[hs["year"] == "all"].pivot(index="pair", columns="hour", values="t").reindex(PAIRS)
    allm = hs[hs["year"] == "all"].pivot(index="pair", columns="hour", values="mean_bps").reindex(PAIRS)
    yr = hs[hs["year"] != "all"]
    same = (yr.merge(allm.stack().rename("m_all").reset_index(), on=["pair", "hour"])
              .assign(s=lambda d: np.sign(d["mean_bps"]) == np.sign(d["m_all"]))
              .groupby(["pair", "hour"])["s"].sum().unstack().reindex(PAIRS))
    fig, axes = plt.subplots(1, 2, figsize=(18, 5.5))
    im = axes[0].imshow(allt.to_numpy(), cmap="RdBu_r", vmin=-5, vmax=5, aspect="auto")
    axes[1].imshow(same.to_numpy(), cmap="viridis", vmin=0, vmax=5, aspect="auto")
    for ax, ttl in zip(axes, ["t-statistic of the hourly mean", "years with the same sign (of 5)"]):
        ax.set_yticks(range(len(PAIRS))); ax.set_yticklabels(PAIRS)
        ax.set_xticks(range(24)); ax.set_xticklabels([F.slot_label(clock, 60 * h).split()[1] for h in range(24)], rotation=90)
        ax.set_title(f"{clock}: {ttl}")
    for i in range(len(PAIRS)):
        for h in range(24):
            axes[1].text(h, i, int(same.iat[i, h]), ha="center", va="center", fontsize=6, color="white")
    fig.colorbar(im, ax=axes[0], fraction=0.02)
    plt.tight_layout(); plt.savefig(OUT / f"01_hourly_{clock}.png", dpi=110); plt.show()
    allt.to_csv(OUT / f"01_hourly_t_{clock}.csv")

## 6. Interval scan: D1–D3 and frozen candidates

In [ ]:
scan = pd.concat([F.scan_panel(p, cfg) for p in panels.values()], ignore_index=True)
scan, cand = F.select_candidates(scan, cfg)
print(f"tests: {len(scan):,} | D1 (BH q<=0.05): {int(scan['D1'].sum())} | D1+D2: {int((scan['D1'] & scan['D2']).sum())} | "
      f"D1+D2+D3: {int((scan['D1'] & scan['D2'] & scan['D3']).sum())} | candidates after de-duplication: {len(cand)}")
by_pair = (scan.assign(D123=scan["D1"] & scan["D2"] & scan["D3"])
               .groupby(["pair", "clock"])[["D1", "D123"]].sum().unstack("clock").reindex(PAIRS))
display(by_pair)
cols = ["id", "pair", "clock", "start_time", "end_time", "minutes", "direction", "mean_bps", "t", "q",
        "years_same_sign", "cost_bps", "n_days", "start", "end"]
display(cand[cols].round(3))
scan[scan["D1"]].to_csv(OUT / "01_scan_D1_survivors.csv", index=False)
frozen = OUT / "candidates_frozen.csv"
cand[cols].to_csv(frozen, index=False, float_format="%.6f")
sha = F.file_sha256(frozen)
print("frozen:", frozen, "sha256", sha)

## 7. Where the candidates sit

Each candidate drawn on its pair's stacked day (discovery data).

In [ ]:
if len(cand):
    n = min(len(cand), 12)
    fig, axes = plt.subplots(int(np.ceil(n / 3)), 3, figsize=(16, 3.2 * np.ceil(n / 3)), squeeze=False)
    for ax, (_, r) in zip(axes.flat, cand.head(n).iterrows()):
        pr = prof[(r["pair"], r["clock"])]
        ax.plot(pr.index / 60, pr["cum_mean_bps"], lw=1)
        ax.axvspan(r["start"] / 60, r["end"] / 60, color="green" if r["direction"] > 0 else "red", alpha=0.2)
        ax.set_title(f"{r['id']} {r['pair']} {r['start_time']}-{r['end_time'].split()[1]} "
                     f"{'long' if r['direction'] > 0 else 'short'} {r['mean_bps']:.2f} bp, t={r['t']:.1f}", fontsize=9)
        ax.set_xticks(range(0, 25, 4)); ax.set_xticklabels([F.slot_label(r["clock"], 60 * h).split()[1] for h in range(0, 25, 4)])
    for ax in axes.flat[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.savefig(OUT / "01_candidates.png", dpi=110); plt.show()
else:
    print("no candidates")

## 8. Global check D0: rotated-day placebos

Each day's 1-minute returns rotated by a random offset (same returns and daily move, time of day destroyed), full pipeline rerun. Takes a few minutes per placebo.

In [ ]:
pl = F.placebo_counts(panels, cfg)
real = int((scan["D1"] & scan["D2"] & scan["D3"]).sum())
p95 = float(np.percentile(pl["D1_D2_D3"], 95))
d0 = "STRUCTURE_FOUND" if real > p95 else "NO_STRUCTURE"
display(pl.describe().loc[["mean", "50%", "max"]])
print(f"real D1-D3 survivors {real} vs placebo 95th percentile {p95:.1f} -> {d0}")
pl.to_csv(OUT / "01_placebo_counts.csv", index=False)

In [ ]:
summary = {"study": "FXS01", "fxs_version": F.FXS_VERSION, "synthetic": SYNTHETIC, "window": [str(START), str(END)],
           "pairs": PAIRS, "tests": int(len(scan)), "D1": int(scan["D1"].sum()), "D1_D2": int((scan["D1"] & scan["D2"]).sum()),
           "D1_D2_D3": real, "candidates": int(len(cand)), "placebo_D1_D2_D3": pl["D1_D2_D3"].tolist(), "D0": d0,
           "candidates_file": "candidates_frozen.csv", "candidates_sha256": sha,
           "config": {k: (list(v) if isinstance(v, tuple) else v) for k, v in asdict(cfg).items()}}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps({k: summary[k] for k in ("tests", "D1", "D1_D2", "D1_D2_D3", "candidates", "D0")}, indent=2))